# Week 13 — Name Generator with a Character-Level LSTM

**Theme:** Recurrent neural networks (RNN) — modeling sequential data

Images don't have an inherent order, but text does — the letters of a word
have to come in the right sequence. An **RNN** (here, its more capable cousin,
the **LSTM**) processes a sequence one step at a time, carrying a "memory"
(hidden state) forward, so what it predicts next depends on everything it has
seen so far.

We'll train a small character-level LSTM to predict "what letter comes next"
in a first name, one character at a time, using ~180 real first names as
training data — then sample brand-new names the model has never seen.

In [ ]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
random.seed(0)

names = ['olivia', 'emma', 'charlotte', 'amelia', 'sophia', 'isabella', 'ava', 'mia', 'evelyn', 'luna', 'harper', 'camila', 'gianna', 'abigail', 'ella', 'elizabeth', 'sofia', 'emily', 'avery', 'mila', 'scarlett', 'eleanor', 'madison', 'layla', 'penelope', 'aria', 'chloe', 'grace', 'nora', 'hazel', 'zoey', 'riley', 'victoria', 'lily', 'hannah', 'lillian', 'addison', 'aubrey', 'ellie', 'stella', 'natalie', 'zoe', 'leah', 'paisley', 'audrey', 'skylar', 'violet', 'claire', 'bella', 'aurora', 'liam', 'noah', 'oliver', 'james', 'elijah', 'mateo', 'theodore', 'henry', 'lucas', 'william', 'benjamin', 'levi', 'sebastian', 'jack', 'ezra', 'michael', 'daniel', 'leo', 'owen', 'samuel', 'hudson', 'alexander', 'asher', 'jayden', 'john', 'joseph', 'wyatt', 'david', 'luke', 'julian', 'grayson', 'gabriel', 'carter', 'isaac', 'jayce', 'anthony', 'dylan', 'luca', 'josiah', 'christopher', 'andrew', 'joshua', 'ryan', 'nathan', 'caleb', 'adrian', 'miles', 'eli', 'nolan', 'christian', 'aaliyah', 'nevaeh', 'serenity', 'valentina', 'kinsley', 'genesis', 'aaliyah', 'peyton', 'alice', 'maya', 'kennedy', 'sadie', 'mackenzie', 'autumn', 'brooklyn', 'savannah', 'allison', 'gabriella', 'sarah', 'alexa', 'ariana', 'alaina', 'naomi', 'elena', 'ivy', 'katherine', 'jasmine', 'ruby', 'vivian', 'reagan', 'piper', 'josephine', 'willow', 'cora', 'adeline', 'melody', 'juniper', 'athena', 'caleb', 'dominic', 'jordan', 'gavin', 'xavier', 'ian', 'jose', 'greyson', 'angel', 'adam', 'kevin', 'brayden', 'robert', 'chase', 'cameron', 'carson', 'axel', 'jonathan', 'charles', 'tyler']
names = sorted(set(names))
print(f"{len(names)} unique training names, e.g.: {names[:8]}")

## 1. Turn characters into numbers

We build a vocabulary of every character that appears in our names, plus two
special symbols: `^` marks "start of name" and `$` marks "end of name" (so the
model can learn *when to stop*).

In [ ]:
chars = sorted(set("".join(names)))
vocab = ["^", "$"] + chars   # index 0 = start, index 1 = end
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for i, ch in enumerate(vocab)}
vocab_size = len(vocab)

print("Vocabulary:", vocab)
print("Vocab size:", vocab_size)

## 2. Define the model

- `Embedding`: turns each character index into a learnable vector
- `LSTM`: processes the sequence of vectors, updating a hidden "memory" state
- `Linear`: turns the LSTM's output at each step into a score for every
  possible next character

In [ ]:
class NameLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)                  # (batch, seq_len) -> (batch, seq_len, embed_dim)
        out, hidden = self.lstm(emb, hidden)  # -> (batch, seq_len, hidden_dim)
        logits = self.fc(out)                 # -> (batch, seq_len, vocab_size)
        return logits, hidden

model = NameLSTM(vocab_size)
print(model)
print("Total parameters:", sum(p.numel() for p in model.parameters()))

## 3. Train: "predict the next character"

For the name `"emma"`, we feed the model `^emma` and ask it to predict
`emma$` — i.e. after `^` predict `e`, after `^e` predict `m`, ..., after
`^emma` predict `$` (stop). We process one name at a time — with only ~180
short names, this stays fast even on CPU.

In [ ]:
def encode(name):
    input_seq = torch.tensor([[char2idx[c] for c in ("^" + name)]])
    target_seq = torch.tensor([char2idx[c] for c in (name + "$")])
    return input_seq, target_seq

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

n_epochs = 30
losses = []
for epoch in range(n_epochs):
    random.shuffle(names)
    total_loss = 0.0
    for name in names:
        input_seq, target_seq = encode(name)
        optimizer.zero_grad()
        logits, _ = model(input_seq)
        loss = criterion(logits.squeeze(0), target_seq)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(names)
    losses.append(avg_loss)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}  avg_loss={avg_loss:.3f}")

plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("Training Loss (average per name)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

## 4. Generate new names

To generate, we feed `^`, sample a character from the model's predicted
probability distribution, feed that character back in, and repeat until the
model predicts `$` (or we hit a max length).

**Temperature** controls randomness: low temperature (e.g. 0.5) sticks close
to the model's most confident guesses (safer, more repetitive); high
temperature (e.g. 1.5) samples more adventurously (more variety, more
nonsense).

In [ ]:
def generate_name(model, temperature=1.0, max_len=15):
    model.eval()
    input_t = torch.tensor([[char2idx["^"]]])
    hidden = None
    result = []
    with torch.no_grad():
        for _ in range(max_len):
            logits, hidden = model(input_t, hidden)
            probs = F.softmax(logits[0, -1] / temperature, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            if idx2char[next_idx] == "$":
                break
            result.append(idx2char[next_idx])
            input_t = torch.tensor([[next_idx]])
    return "".join(result).capitalize()

training_set = set(names)
print("Generated names (temperature=0.8):")
for _ in range(15):
    name = generate_name(model, temperature=0.8)
    tag = " (matches training data)" if name.lower() in training_set else " (new!)"
    print(" -", name, tag)

In [ ]:
print("Low temperature (0.5) -- more conservative:")
print([generate_name(model, temperature=0.5) for _ in range(8)])

print("High temperature (1.5) -- more random:")
print([generate_name(model, temperature=1.5) for _ in range(8)])

## Try it yourself

1. **Train longer.** Bump `n_epochs` to 60 — do the generated names look more
   name-like? Do you see it start memorizing the training set instead of
   generalizing?
2. **Bigger hidden state.** Try `NameLSTM(vocab_size, hidden_dim=128)` — does
   it change how quickly the loss drops?
3. **Different data.** Replace the `names` list with a list of Pokémon names,
   city names, or your own made-up word list, retrain, and see what kind of
   words come out.
4. **Compare to Week 12.** Both this week and Week 12 turn discrete symbols
   (characters here, words there) into vectors the network can use — what's
   different about what's being predicted in each case?